# Bernstein-Vazirani Algorithm

Recover a hidden four-bit string with one quantum oracle call. The oracle computes the bitwise inner product of the input and hidden string, modulo 2.

Run the cells from top to bottom in a fresh Python kernel. See the [repository README](../README.md) for environment setup.

**Bit ordering:** Qiskit displays bit strings with the highest-index bit on the left and bit 0 on the right. Ideal simulator results are used here; shot counts can fluctuate when several outcomes have nonzero probability.

In [ ]:
# Imports needed to run this notebook independently.
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram

## Recovering the hidden string

A CNOT targets the auxiliary qubit for each 1 in the hidden string. Reversing the string maps its rightmost bit to q0. The measured result should exactly match the generated hidden string, including leading zeros.

In [ ]:
n  = 4

s = ''.join([str(random.randint(0,1)) for _ in range(n)])

def oracle(s):
    m = len(s)
    qc = QuantumCircuit(m + 1)
    for i , c in enumerate(reversed(s)):
        if c == "1":
            qc.cx(i , m)  # The auxiliary index comes from this oracle's string length.
    return qc

orc = oracle(s)

qc = QuantumCircuit(n + 1, n)
qc.x(n)
qc.h(range(n + 1))
qc.compose(orc, inplace=True)
qc.h(range(n))
qc.measure(range(n), range(n))

sm = Aer.get_backend("qasm_simulator")
tc = transpile(qc , sm)
r = sm.run(tc , shots = 1024).result().get_counts()

display(qc.draw("mpl"))
display(plot_histogram(r))

maxa = max(r , key=r.get)
print("Hidden string:", s)
print("Recovered string:", maxa)